In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

In [2]:
val packageVersion = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % packageVersion)  // use programmatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

packageVersion: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset, Row, Column}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension
import org.dataprov.dp.SemiWhyProvenanceBuilder
import org.dataprov.dp.DisplayStringProvenanceBuilder
import org.dataprov.dp.ProvenanceBuilder
import org.dataprov.dp.BooleanProvenanceBuilder
import org.dataprov.dp.FullWhyProvenanceBuilder
import org.dataprov.dp.LightWhyProvenanceBuilder


import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset, Row, Column}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._
import org.dataprov.dp.LogicalPlanWithProvenance
import org.dataprov.dp.ProvenanceExtension
import org.dataprov.dp.SemiWhyProvenanceBuilder
import org.dataprov.dp.DisplayStringProvenanceBuilder
import org.dataprov.dp.ProvenanceBuilder
import org.dataprov.dp.BooleanProvenanceBuilder
import org.dataprov.dp.FullWhyProvenanceBuilder
import org.dataprov.dp.LightWhyProvenanceBuilder

# Predefined provenance builder 


The Provenance Operators can be customed :
- using a custom provenance builder (must be set in extension, cannot be overridden via config)


Example using a custom provenance builder (must be set in extension, cannot be overridden via config) :

```
val spark = sparkWithProvenance(
provenanceBuilder = TokenArrayProvenanceBuilder,
appName = "custom-provenance-test"
)
```

In [4]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook-why-provenance")
  .withExtensions(
    new ProvenanceExtension(
      // can be customized with different operators and builders
      provenanceBuilder = SemiWhyProvenanceBuilder
    )
  )
  // .config("spark.jars", s"../target/scala-2.13/dp-spark_2.13-$packageVersion.jar")
  // .config("spark.sql.extensions", "org.dataprov.dp.ProvenanceExtension")
  .config("spark.provenance.enabled", "true")
  .getOrCreate()

println(s"Spark provenance enabled: ${spark.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/02 17:27:54 INFO SparkContext: Running Spark version 4.1.1
26/06/02 17:27:54 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/06/02 17:27:54 INFO SparkContext: Java version 17.0.10+7
26/06/02 17:27:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/02 17:27:54 INFO ResourceUtils: ==============================================================
26/06/02 17:27:54 INFO ResourceUtils: No custom resources configured for spark.driver.
26/06/02 17:27:54 INFO ResourceUtils: ==============================================================
26/06/02 17:27:54 INFO SparkContext: Submitted application: notebook-why-provenance
26/06/02 17:27:54 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/06/02 17:27:54 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/06/02 17:27:54 INFO SecurityManager: Changing view

Spark provenance enabled: true


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@663168bd

In [5]:
val df0: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df0.show()

val dfWithProvenance = df0.addProvenanceColumn(col("A"))
dfWithProvenance.printSchema()
dfWithProvenance.show(false)

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: string (nullable = true)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a              |
|d  |b  |e  |d              |
|f  |g  |e  |f              |
+---+---+---+---------------+



df0: DataFrame = [A: string, B: string ... 1 more field]
dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [6]:
val df2WithProvenance : DataFrame = dfWithProvenance
    .select("A", "B")
    .join(dfWithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show(false)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|a  |b  |e  |[a, d]         |
|d  |b  |c  |[d, a]         |
|d  |b  |e  |[d]            |
|f  |g  |e  |[f]            |
+---+---+---+---------------+



df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [7]:
dfWithProvenance.createOrReplaceTempView("df_with_prov")
val df3WithProvenance: DataFrame = spark.sql("""
    SELECT A, B, t1.C
    FROM (
        SELECT A, C
        FROM df_with_prov
    ) AS t1
    JOIN (
        SELECT B, C
        FROM df_with_prov
    ) AS t2
    ON t1.C = t2.C
    ORDER BY A, B, t1.C
""")

df3WithProvenance.show(false)

val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show(false)

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show(false)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|d  |b  |e  |[d]            |
|d  |g  |e  |[d, f]         |
|f  |b  |e  |[f, d]         |
|f  |g  |e  |[f]            |
+---+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|a  |b  |e  |[a, d]         |
|d  |b  |c  |[d, a]         |
|d  |b  |e  |[d]            |
|d  |g  |e  |[d, f]         |
|f  |b  |e  |[f, d]         |
|f  |g  |e  |[f]            |
+---+---+---+---------------+

+---+---+---------------+
|A  |C  |_provenance_tag|
+---+---+---------------+
|a  |c  |[a]            |
|a  |e  |[a, d]         |
|d  |c  |[d, a]         |
|d  |e  |[d]            |
|f  |e  |[f]            |
+---+---+---------------+



df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df4WithProvenance: Dataset[Row] = [A: string, B: string ... 2 more fields]
df5WithProvenance: Dataset[Row] = [A: string, C: string ... 1 more field]

In [8]:
val df6WithProvenance = dfWithProvenance.groupBy("C").agg(collect_list("A").as("A_list")).orderBy("C")
df6WithProvenance.show(false)   

+---+------+---------------+
|C  |A_list|_provenance_tag|
+---+------+---------------+
|c  |[a]   |[a]            |
|e  |[d, f]|[d, f]         |
+---+------+---------------+



df6WithProvenance: Dataset[Row] = [C: string, A_list: array<string> ... 1 more field]

# Custom provenance builder

Test with the example taken from the research

In [9]:
import org.apache.spark.sql.types.{DataType, StringType}

// Example of a custom string-based provenance builder
object CustomStringProvenanceBuilder extends ProvenanceBuilder {

  val joinOperator: String = " * "
  val aggregateOperator: String = " + "
  val distinctOperator: String = " + "
  
  override val provType: DataType = StringType

  override def single(attr: Attribute): Expression = {
    Concat(Seq(Literal("["), Cast(attr, StringType), Literal("]")))
  }

  override def join(
      left: Attribute,
      right: Attribute
  ): Expression = {
    val leftCast = Cast(left, StringType)
    val rightCast = Cast(right, StringType)

    val matchedTag = If(
      And(IsNotNull(left), IsNotNull(right)),
      Concat(Seq(leftCast, Literal(joinOperator), rightCast)),
      Cast(Literal(null), StringType)
    )

    val leftOnly = If(IsNotNull(left), leftCast, Cast(Literal(null), StringType))
    val rightOnly = If(IsNotNull(right), rightCast, Cast(Literal(null), StringType))

    Coalesce(Seq(matchedTag, leftOnly, rightOnly))
  }

  override def distinct(
      attr: Attribute
  ): Expression = {
    val collectSetExpr = AggregateExpression(
      CollectSet(Cast(attr, StringType)),
      Complete,
      isDistinct = false
    )
    ConcatWs(Seq(Literal(distinctOperator), collectSetExpr))
  }

  override def aggregate(
      attr: Attribute
  ): Expression = {
    val collectSetExpr = AggregateExpression(
      CollectSet(Cast(attr, StringType)),
      Complete,
      isDistinct = false
    )
    ConcatWs(Seq(Literal(aggregateOperator), collectSetExpr))
  }
}

// Stop current SparkSession so extensions are applied on a fresh one
spark.stop()

val sparkCustom = SparkSession.builder()
  .master("local[*]")
  .appName("notebook-custom-string-provenance")
  .withExtensions(
    new ProvenanceExtension(
      provenanceBuilder = CustomStringProvenanceBuilder
    )
  )
  .config("spark.provenance.enabled", "true")
  .getOrCreate()

println(s"Spark provenance enabled: ${sparkCustom.conf.get("spark.provenance.enabled")}")
sparkCustom.sparkContext.setLogLevel("ERROR")

Spark provenance enabled: true


import org.apache.spark.sql.types.{DataType, StringType}
defined object CustomStringProvenanceBuilder
sparkCustom: SparkSession = org.apache.spark.sql.classic.SparkSession@7cfb8a50

In [10]:

val df1: DataFrame = sparkCustom.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df1.show()

val df1WithProvenance = df1.addProvenanceColumn(col("A"))
df1WithProvenance.printSchema()
df1WithProvenance.show(false)

val df2WithProvenance : DataFrame = df1WithProvenance
    .select("A", "B")
    .join(df1WithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show(false)

df1WithProvenance.createOrReplaceTempView("df_with_prov")
val df3WithProvenance: DataFrame = sparkCustom.sql("""
    SELECT A, B, t1.C
    FROM (
        SELECT A, C
        FROM df_with_prov
    ) AS t1
    JOIN (
        SELECT B, C
        FROM df_with_prov
    ) AS t2
    ON t1.C = t2.C
    ORDER BY A, B, t1.C
""")

df3WithProvenance.show(false)



val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show(false)

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show(false)

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: string (nullable = true)

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a              |
|d  |b  |e  |d              |
|f  |g  |e  |f              |
+---+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a * a          |
|a  |b  |e  |a * d          |
|d  |b  |c  |d * a          |
|d  |b  |e  |d * d          |
|f  |g  |e  |f * f          |
+---+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |a * a          |
|d  |b  |e  |d * d          |
|d  |g  |e  |d * f          |
|f  |b  |e  |f * d          |
|f  |g  |e  |f * f          |
+---+---+---+--------------

df1: DataFrame = [A: string, B: string ... 1 more field]
df1WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df4WithProvenance: Dataset[Row] = [A: string, B: string ... 2 more fields]
df5WithProvenance: Dataset[Row] = [A: string, C: string ... 1 more field]

In [11]:
val df6WithProvenance = df1WithProvenance
  .groupBy("B")
  .agg(
    collect_list("A").as("A_list"),
    collect_list("C").as("C_list")
)
  .orderBy("B")

df6WithProvenance.show(false)

+---+------+------+---------------+
|B  |A_list|C_list|_provenance_tag|
+---+------+------+---------------+
|b  |[a, d]|[c, e]|a + d          |
|g  |[f]   |[e]   |f              |
+---+------+------+---------------+



df6WithProvenance: Dataset[Row] = [B: string, A_list: array<string> ... 2 more fields]

# Why-provenance (witness tuples only)

In [12]:
SparkSession.getActiveSession.foreach(_.stop())
SparkSession.getDefaultSession.foreach(_.stop())
try { spark.stop() } catch { case _: Throwable => () }
try { sparkCustom.stop() } catch { case _: Throwable => () }

val sparkWhy = SparkSession.builder()
  .master("local[*]")
  .appName("notebook-why-provenance")
  .withExtensions(
    new ProvenanceExtension(
      provenanceBuilder = SemiWhyProvenanceBuilder
    )
  )
  .config("spark.provenance.enabled", "true")
  .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
sparkWhy.sparkContext.setLogLevel("ERROR")

Spark provenance enabled: true


sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@272ff7ca

In [13]:
val dfWhy = sparkWhy.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

val dfWhyProv = dfWhy.addProvenanceColumn(col("A"))

val whyResult = dfWhyProv
  .select("A", "B")
  .join(dfWhyProv.select("B", "C"), "B")
  .select("A", "C")
  .distinct()
  .orderBy("A", "C")

whyResult.printSchema()
whyResult.show(false)

val testMixedWhy = whyResult.join(dfWhyProv.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")

testMixedWhy.show(false)

val test = whyResult.agg(collect_set("C").as("C_set"))
test.show(false)

root
 |-- A: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---+---+---------------+
|A  |C  |_provenance_tag|
+---+---+---------------+
|a  |c  |[a]            |
|a  |e  |[a, d]         |
|d  |c  |[d, a]         |
|d  |e  |[d]            |
|f  |e  |[f]            |
+---+---+---------------+

+---+---+---+---------------+
|A  |B  |C  |_provenance_tag|
+---+---+---+---------------+
|a  |b  |c  |[a]            |
|a  |b  |e  |[a, d]         |
|a  |g  |e  |[a, d, f]      |
|d  |b  |c  |[d, a]         |
|d  |b  |e  |[d]            |
|d  |g  |e  |[d, f]         |
|f  |b  |e  |[f, d]         |
|f  |g  |e  |[f]            |
+---+---+---+---------------+

+------+---------------+
|C_set |_provenance_tag|
+------+---------------+
|[e, c]|[d, a, f]      |
+------+---------------+



dfWhy: DataFrame = [A: string, B: string ... 1 more field]
dfWhyProv: DataFrame = [A: string, B: string ... 2 more fields]
whyResult: Dataset[Row] = [A: string, C: string ... 1 more field]
testMixedWhy: Dataset[Row] = [A: string, B: string ... 2 more fields]
test: DataFrame = [C_set: array<string>, _provenance_tag: array<string>]

In [14]:
SparkSession.getActiveSession.foreach(_.stop())
SparkSession.getDefaultSession.foreach(_.stop())

// Create SparkSession
val sparkWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        // can be customized with different operators and builders
        new ProvenanceExtension(provenanceBuilder = SemiWhyProvenanceBuilder)
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

println(s"Spark provenance enabled: ${sparkWhy.conf.get("spark.provenance.enabled")}")

// Set log level to ERROR to reduce verbosity
sparkWhy.sparkContext.setLogLevel("ERROR")

// Set the name of the provenance column to "why_prov" 
// (optional, since the default is "_provenance_tag")
// sparkWhy.conf.set("spark.provenance.columnName", "why_prov")

Spark provenance enabled: true


sparkWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@615f069d

In [15]:
val df: DataFrame = sparkWhy.createDataFrame(
    Seq(
        ("a1","A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("a2","A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("a3","A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("b1","B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("b2","B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("b3","B", Date.valueOf("2026-01-16"), 120.0, 25),
        ("c1","C", Date.valueOf("2026-01-17"), 80.0, 60),
        ("c2","C", Date.valueOf("2026-01-18"), 82.0, 50),
        ("d1","D", Date.valueOf("2026-01-16"), 50.0, 10)
    )
).toDF("id", "product", "date", "price", "sales")

df.show()

+---+-------+----------+-----+-----+
| id|product|      date|price|sales|
+---+-------+----------+-----+-----+
| a1|      A|2026-01-15| 10.0|   90|
| a2|      A|2026-01-16| 10.0|  120|
| a3|      A|2026-01-17|  5.0|  300|
| b1|      B|2026-01-15|100.0|   20|
| b2|      B|2026-01-16|100.0|   30|
| b3|      B|2026-01-16|120.0|   25|
| c1|      C|2026-01-17| 80.0|   60|
| c2|      C|2026-01-18| 82.0|   50|
| d1|      D|2026-01-16| 50.0|   10|
+---+-------+----------+-----+-----+



df: DataFrame = [id: string, product: string ... 3 more fields]

In [16]:
val dfWithProv : DataFrame = df.addProvenanceColumn
dfWithProv.show(false)

val dfWithProv2 : DataFrame = df.addProvenanceColumn(col("product"))
dfWithProv2.show(false)

val dfWithProv3 : DataFrame = df.addProvenanceColumn(col("id"))
dfWithProv3.show(false)

+---+-------+----------+-----+-----+------------------------------------+
|id |product|date      |price|sales|_provenance_tag                     |
+---+-------+----------+-----+-----+------------------------------------+
|a1 |A      |2026-01-15|10.0 |90   |ada33396-6af0-42c1-8037-3374b6c9a314|
|a2 |A      |2026-01-16|10.0 |120  |9fdc57f3-8069-4e30-b703-d6b33f94fcad|
|a3 |A      |2026-01-17|5.0  |300  |56a90128-c494-4adf-8462-03ba7686d9fd|
|b1 |B      |2026-01-15|100.0|20   |e0ec05c9-727b-4761-9a6f-07f7f7bd1c74|
|b2 |B      |2026-01-16|100.0|30   |934179ce-1d2b-4030-9b85-2a05f521bb27|
|b3 |B      |2026-01-16|120.0|25   |6f92feff-c096-4150-bd4e-f33cd6cf0cb0|
|c1 |C      |2026-01-17|80.0 |60   |2ee4391b-cbc8-4242-a035-76b7cff9fd58|
|c2 |C      |2026-01-18|82.0 |50   |59d8edf7-3fea-4da9-996b-129e62458b19|
|d1 |D      |2026-01-16|50.0 |10   |a38f9fea-5853-45ea-81fa-46edddd148f9|
+---+-------+----------+-----+-----+------------------------------------+

+---+-------+----------+-----+-----+-

dfWithProv: DataFrame = [id: string, product: string ... 4 more fields]
dfWithProv2: DataFrame = [id: string, product: string ... 4 more fields]
dfWithProv3: DataFrame = [id: string, product: string ... 4 more fields]

In [17]:
val distinctProducts = dfWithProv3.select("product").distinct()
distinctProducts.show(false)

val distinctDates = dfWithProv3.select("date", "product").groupBy("date").agg(collect_set("product").as("products"))
distinctDates.show(false)

val joined = dfWithProv3.join(distinctProducts, "product")
joined.show(false)

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|A      |[a3]           |
|B      |[b3]           |
|C      |[c2]           |
|D      |[d1]           |
+-------+---------------+

+----------+---------+----------------+
|date      |products |_provenance_tag |
+----------+---------+----------------+
|2026-01-15|[A, B]   |[a1, b1]        |
|2026-01-16|[A, D, B]|[a2, d1, b3, b2]|
|2026-01-17|[A, C]   |[c1, a3]        |
|2026-01-18|[C]      |[c2]            |
+----------+---------+----------------+

+-------+---+----------+-----+-----+---------------+
|product|id |date      |price|sales|_provenance_tag|
+-------+---+----------+-----+-----+---------------+
|A      |a1 |2026-01-15|10.0 |90   |[a1, a3]       |
|A      |a2 |2026-01-16|10.0 |120  |[a2, a3]       |
|A      |a3 |2026-01-17|5.0  |300  |[a3]           |
|B      |b1 |2026-01-15|100.0|20   |[b1, b3]       |
|B      |b2 |2026-01-16|100.0|30   |[b2, b3]       |
|B      |b3 |2026-01-16|120.0|25   |[b3]      

distinctProducts: Dataset[Row] = [product: string, _provenance_tag: array<string>]
distinctDates: DataFrame = [date: date, products: array<string> ... 1 more field]
joined: DataFrame = [product: string, id: string ... 4 more fields]

In [18]:
// Query with groupBy date and collect_set of products
val query = dfWithProv3
  .select("product", "date")
  .groupBy("date")
  .agg(collect_set("product").as("products"))

query.show(false)

+----------+---------+----------------+
|date      |products |_provenance_tag |
+----------+---------+----------------+
|2026-01-15|[A, B]   |[a1, b1]        |
|2026-01-16|[A, D, B]|[a2, d1, b3, b2]|
|2026-01-17|[A, C]   |[c1, a3]        |
|2026-01-18|[C]      |[c2]            |
+----------+---------+----------------+



query: DataFrame = [date: date, products: array<string> ... 1 more field]

# Obtain the rows to keep from result lines

In [ ]:
// Select the row(s) of interest from the query result (e.g., for specific dates)
val rowOfInterest = query.filter(col("date") =!= "2026-01-16")
rowOfInterest.show(false)

// Extract provenance tokens from the selected result rows
val witnessTags = rowOfInterest
  .select(explode(col("_provenance_tag")).as("prov_tag"))
  .distinct()

// Disable provenance for the next operations to avoid generating new tags and keep only the original ones
// sparkWhy.conf.set("spark.provenance.enabled", "false")

//Keep only original rows that contributed to the row(s) of interest
val rowsToKeep = dfWithProv3
  .join(
    witnessTags,
    dfWithProv3.col("_provenance_tag") === witnessTags.col("prov_tag"),
    "inner"
  ).drop("prov_tag")
  //.removeProvenanceColumn
  
rowsToKeep.show(false)

+----------+--------+---------------+
|date      |products|_provenance_tag|
+----------+--------+---------------+
|2026-01-15|[A, B]  |[a1, b1]       |
|2026-01-17|[A, C]  |[c1, a3]       |
|2026-01-18|[C]     |[c2]           |
+----------+--------+---------------+

+---+-------+----------+-----+-----+---------------+
|id |product|date      |price|sales|_provenance_tag|
+---+-------+----------+-----+-----+---------------+
|a1 |A      |2026-01-15|10.0 |90   |[a1, b1]       |
|a3 |A      |2026-01-17|5.0  |300  |[a3, c1]       |
|b1 |B      |2026-01-15|100.0|20   |[b1, a1]       |
|c1 |C      |2026-01-17|80.0 |60   |[c1, a3]       |
|c2 |C      |2026-01-18|82.0 |50   |[c2]           |
+---+-------+----------+-----+-----+---------------+



rowOfInterest: Dataset[Row] = [date: date, products: array<string> ... 1 more field]
witnessTags: Dataset[Row] = [prov_tag: string, _provenance_tag: array<string>]
rowsToKeep: DataFrame = [id: string, product: string ... 4 more fields]

In [20]:
val baseDF = sparkWhy.createDataFrame(
    Seq(
        ("a1","A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("a2","A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("a3","A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("b1","B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("b2","B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("b3","B", Date.valueOf("2026-01-16"), 120.0, 25),
        ("c1","C", Date.valueOf("2026-01-17"), 80.0, 60),
        ("c2","C", Date.valueOf("2026-01-18"), 82.0, 50),
        ("d1","D", Date.valueOf("2026-01-16"), 50.0, 10),
        ("e1","E", Date.valueOf("2026-01-15"), 60.0, 15)
    )
    
).toDF("id", "product", "date", "price", "sales")

val multiplicateur = 1000 
val generateurDF = sparkWhy.range(0, multiplicateur).toDF("rep_id")

// 3. On fait un produit cartésien (Cross Join) pour multiplier les lignes
// Attention : 9 lignes * 100 000 = 900 000 lignes
val bigDF = baseDF.crossJoin(generateurDF)
  .withColumn("id", concat(col("id"), lit("_"), col("rep_id"))) // On rend l'ID unique
  .drop("rep_id")

bigDF.show(10) // Affiche les 10 premières lignes pour vérifier le résultat

val bigDFWithProv = bigDF.addProvenanceColumn(col("id"))
bigDFWithProv.show(10)

+----+-------+----------+-----+-----+
|  id|product|      date|price|sales|
+----+-------+----------+-----+-----+
|a1_0|      A|2026-01-15| 10.0|   90|
|a2_0|      A|2026-01-16| 10.0|  120|
|a3_0|      A|2026-01-17|  5.0|  300|
|b1_0|      B|2026-01-15|100.0|   20|
|b2_0|      B|2026-01-16|100.0|   30|
|b3_0|      B|2026-01-16|120.0|   25|
|c1_0|      C|2026-01-17| 80.0|   60|
|c2_0|      C|2026-01-18| 82.0|   50|
|d1_0|      D|2026-01-16| 50.0|   10|
|e1_0|      E|2026-01-15| 60.0|   15|
+----+-------+----------+-----+-----+
only showing top 10 rows
+----+-------+----------+-----+-----+---------------+
|  id|product|      date|price|sales|_provenance_tag|
+----+-------+----------+-----+-----+---------------+
|a1_0|      A|2026-01-15| 10.0|   90|           a1_0|
|a2_0|      A|2026-01-16| 10.0|  120|           a2_0|
|a3_0|      A|2026-01-17|  5.0|  300|           a3_0|
|b1_0|      B|2026-01-15|100.0|   20|           b1_0|
|b2_0|      B|2026-01-16|100.0|   30|           b2_0|
|b3_0|     

baseDF: DataFrame = [id: string, product: string ... 3 more fields]
multiplicateur: Int = 1000
generateurDF: DataFrame = [rep_id: bigint]
bigDF: DataFrame = [id: string, product: string ... 3 more fields]
bigDFWithProv: DataFrame = [id: string, product: string ... 4 more fields]

# Compare Full-Why-Provenance, Light-Why-Provenance and Semi-Light-Provenance

We propose implementing multiple builders tailored to different levels of provenance granularity:

1. **Full-Why-Provenance (Complete Lineage Track):** Retains all provenance tags, including those from `DISTINCT` and `AGGREGATE` operations. This aligns with the $\mathcal{P}(\mathcal{P}(X))$ algebraic framework $(\mathcal{P}(\mathcal{P}(X)), \cup, \Cup, \emptyset, \{\emptyset\})$.
   * *Distinct:* All alternative tags are preserved in a subset, offering a "choice" among contributors.
   * *Aggregate:* Every individual tag is kept, acknowledging each as a contributor.

2. **Semi-Why-Provenance (Current Baseline):** Retains all tags for aggregates but minimizes them for distinct operations.
   * *Distinct:* Keeps only the smallest subset of tags.
   * *Aggregate:* Keeps all individual tags as in Full-Why-Provenace.

3. **Light-Why-Provenance (Minimal Footprint):** Retains only a single representative occurrence for distinct operations, and either discards aggregate tags entirely or preserves just a single witness occurrence.
   * *Distinct:* Keeps only the smallest subset of tags. 
   * *Aggregate:* Keeps only the smallest subset of tags.

In [21]:
def queryProductsByDate(df: DataFrame): DataFrame = {
  df.groupBy("product")
    .agg(sum(col("price") * col("sales")).as("total_revenue"))
    .orderBy("product")
} 


def queryDistinctProducts(df: DataFrame): DataFrame = {
  df.select("product").distinct()
}

def queryJoinProducts(df: DataFrame): DataFrame = {
  df.select("product", "date")
    .filter(col("date") === "2026-01-16")
    .join(df.select("product", "sales"), "product")
    .select("product", "date", "sales")
    .orderBy("product", "date", "sales")
}

def createDF(spark : SparkSession): DataFrame = {
  val baseDF = spark.createDataFrame(
    Seq(
        ("a1","A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("a2","A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("a3","A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("b1","B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("b2","B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("b3","B", Date.valueOf("2026-01-16"), 120.0, 25),
        ("c1","C", Date.valueOf("2026-01-17"), 80.0, 60),
        ("c2","C", Date.valueOf("2026-01-18"), 82.0, 50),
        ("d1","D", Date.valueOf("2026-01-16"), 50.0, 10),
        ("e1","E", Date.valueOf("2026-01-15"), 60.0, 15)
    )  
  ).toDF("id", "product", "date", "price", "sales")
  baseDF
}

def createDFWithProvenance(spark: SparkSession): DataFrame = {
  val baseDF = createDF(spark)
  baseDF.addProvenanceColumn(col("id"))
}

defined function queryProductsByDate
defined function queryDistinctProducts
defined function queryJoinProducts
defined function createDF
defined function createDFWithProvenance

In [22]:
SparkSession.getActiveSession.foreach(_.stop())
SparkSession.getDefaultSession.foreach(_.stop())

// Create SparkSession
val sparkFullWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        // can be customized with different operators and builders
        new ProvenanceExtension(provenanceBuilder = FullWhyProvenanceBuilder)
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

// Set log level to ERROR to reduce verbosity
sparkFullWhy.sparkContext.setLogLevel("ERROR")

// Set the name of the provenance column to "why_prov" 
// (optional, since the default is "_provenance_tag")
// sparkFullWhy.conf.set("spark.provenance.columnName", "why_prov")

sparkFullWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@756cfcc2

In [23]:
val DFFullWhy = createDFWithProvenance(sparkFullWhy)

queryProductsByDate(DFFullWhy).show(false)

queryDistinctProducts(DFFullWhy).show(false)

queryJoinProducts(DFFullWhy).show(false)

+-------+-------------+---------------+
|product|total_revenue|_provenance_tag|
+-------+-------------+---------------+
|A      |3600.0       |[a2, a1, a3]   |
|B      |8000.0       |[b3, b1, b2]   |
|C      |8900.0       |[c2, c1]       |
|D      |500.0        |[d1]           |
|E      |900.0        |[e1]           |
+-------+-------------+---------------+

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|A      |[a2, a1, a3]   |
|B      |[b3, b1, b2]   |
|C      |[c2, c1]       |
|D      |[d1]           |
|E      |[e1]           |
+-------+---------------+

+-------+----------+-----+---------------+
|product|date      |sales|_provenance_tag|
+-------+----------+-----+---------------+
|A      |2026-01-16|90   |[a2, a1]       |
|A      |2026-01-16|120  |[a2]           |
|A      |2026-01-16|300  |[a2, a3]       |
|B      |2026-01-16|20   |[b3, b1]       |
|B      |2026-01-16|20   |[b2, b1]       |
|B      |2026-01-16|25   |[b3]           |
|B      |2026-01-

DFFullWhy: DataFrame = [id: string, product: string ... 4 more fields]

In [24]:
SparkSession.getActiveSession.foreach(_.stop())
SparkSession.getDefaultSession.foreach(_.stop())

// Create SparkSession
val sparkLightWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        // can be customized with different operators and builders
        new ProvenanceExtension(provenanceBuilder = LightWhyProvenanceBuilder)
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

// Set log level to ERROR to reduce verbosity
sparkLightWhy.sparkContext.setLogLevel("ERROR")

// Set the name of the provenance column to "why_prov" 
// (optional, since the default is "_provenance_tag")
// sparkLightWhy.conf.set("spark.provenance.columnName", "why_prov")

sparkLightWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@3fafc964

In [25]:
val DFLightWhy = createDFWithProvenance(sparkLightWhy)

queryProductsByDate(DFLightWhy).show(false)

queryDistinctProducts(DFLightWhy).show(false)

queryJoinProducts(DFLightWhy).show(false)

+-------+-------------+---------------+
|product|total_revenue|_provenance_tag|
+-------+-------------+---------------+
|A      |3600.0       |[a3]           |
|B      |8000.0       |[b3]           |
|C      |8900.0       |[c2]           |
|D      |500.0        |[d1]           |
|E      |900.0        |[e1]           |
+-------+-------------+---------------+

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|A      |[a3]           |
|B      |[b3]           |
|C      |[c2]           |
|D      |[d1]           |
|E      |[e1]           |
+-------+---------------+

+-------+----------+-----+---------------+
|product|date      |sales|_provenance_tag|
+-------+----------+-----+---------------+
|A      |2026-01-16|90   |[a2, a1]       |
|A      |2026-01-16|120  |[a2]           |
|A      |2026-01-16|300  |[a2, a3]       |
|B      |2026-01-16|20   |[b3, b1]       |
|B      |2026-01-16|20   |[b2, b1]       |
|B      |2026-01-16|25   |[b3]           |
|B      |2026-01-

DFLightWhy: DataFrame = [id: string, product: string ... 4 more fields]

In [26]:
SparkSession.getActiveSession.foreach(_.stop())
SparkSession.getDefaultSession.foreach(_.stop())

// Create SparkSession
val sparkSemiWhy = SparkSession.builder()
    .appName("notebook-demo-why-provenance")
    .master("local[*]")
    .withExtensions(
        // can be customized with different operators and builders
        new ProvenanceExtension(provenanceBuilder = SemiWhyProvenanceBuilder)
    )
    .config("spark.provenance.enabled", "true")
    .getOrCreate()

// Set log level to ERROR to reduce verbosity
sparkSemiWhy.sparkContext.setLogLevel("ERROR")

// Set the name of the provenance column to "why_prov" 
// (optional, since the default is "_provenance_tag")
// sparkSemiWhy.conf.set("spark.provenance.columnName", "why_prov")

sparkSemiWhy: SparkSession = org.apache.spark.sql.classic.SparkSession@4c7d5350

In [27]:
val DFSemiWhy = createDFWithProvenance(sparkSemiWhy)

queryProductsByDate(DFSemiWhy).show(false)

queryDistinctProducts(DFSemiWhy).show(false)

queryJoinProducts(DFSemiWhy).show(false)

+-------+-------------+---------------+
|product|total_revenue|_provenance_tag|
+-------+-------------+---------------+
|A      |3600.0       |[a2, a1, a3]   |
|B      |8000.0       |[b3, b1, b2]   |
|C      |8900.0       |[c2, c1]       |
|D      |500.0        |[d1]           |
|E      |900.0        |[e1]           |
+-------+-------------+---------------+

+-------+---------------+
|product|_provenance_tag|
+-------+---------------+
|A      |[a3]           |
|B      |[b3]           |
|C      |[c2]           |
|D      |[d1]           |
|E      |[e1]           |
+-------+---------------+

+-------+----------+-----+---------------+
|product|date      |sales|_provenance_tag|
+-------+----------+-----+---------------+
|A      |2026-01-16|90   |[a2, a1]       |
|A      |2026-01-16|120  |[a2]           |
|A      |2026-01-16|300  |[a2, a3]       |
|B      |2026-01-16|20   |[b3, b1]       |
|B      |2026-01-16|20   |[b2, b1]       |
|B      |2026-01-16|25   |[b3]           |
|B      |2026-01-

DFSemiWhy: DataFrame = [id: string, product: string ... 4 more fields]